# Phase 6 — Notebook 1: Data Validation and Monitoring

This notebook demonstrates the input validation layer that guards both prediction endpoints. It also inspects the existing prediction audit log to verify governance compliance.

**Scope:**
- Load and inspect the enriched feature schemas
- Run `validate_payload` against valid and invalid examples for both models
- Show all three check types in action: missing values, numeric range violations, unseen categories
- Parse and summarise the live audit log from Phase 5

In [1]:
import json
import re
import sys
from pathlib import Path

import pandas as pd

# Resolve paths relative to the notebook location
PHASE6_DIR = Path(".").resolve()
REPO_ROOT = PHASE6_DIR.parent.parent
MODEL_FILES = REPO_ROOT / "Model Files"
LOG_PATH = REPO_ROOT / "Notebooks" / "Phase 5 — Deployment and API Integration" / "logs" / "predictions.log"

# Make validate.py importable
if str(PHASE6_DIR) not in sys.path:
    sys.path.insert(0, str(PHASE6_DIR))
from validate import validate_payload

print("validate_payload imported successfully")

validate_payload imported successfully


## 1. Feature Schema Overview

In [2]:
with open(MODEL_FILES / "patient_risk_model_feature_schema.json") as f:
    risk_schema = json.load(f)

with open(MODEL_FILES / "claim_outcome_model_feature_schema.json") as f:
    claim_schema = json.load(f)

def schema_summary(schema: dict) -> pd.DataFrame:
    rows = []
    for feat in schema["features"]:
        c = feat.get("constraints", {})
        rows.append({
            "Feature": feat["name"],
            "Type": feat["type"],
            "Required": feat.get("required", False),
            "min_value": c.get("min_value", ""),
            "max_value": c.get("max_value", ""),
            "allowed_values": ", ".join(str(v) for v in c["allowed_values"]) if "allowed_values" in c else "",
        })
    return pd.DataFrame(rows)

print(f"=== Patient Risk Schema (v{risk_schema['version']}) ===")
display(schema_summary(risk_schema))

print(f"\n=== Claim Outcome Schema (v{claim_schema['version']}) ===")
display(schema_summary(claim_schema))

=== Patient Risk Schema (v1.0) ===


,Feature,Type,Required,min_value,max_value,allowed_values
0,chronic_flag,int8,True,,,"0, 1"
1,gender,category,True,,,"M, F"
2,visit_frequency,int8,True,1,100,



=== Claim Outcome Schema (v1.0) ===


,Feature,Type,Required,min_value,max_value,allowed_values
0,age,int8,True,0,120,
1,gender,category,True,,,"M, F"
2,city,category,True,,,"Bangalore, Chennai, Delhi, Hyderabad, Mumbai, ..."
3,insurance_provider,category,True,,,"CareOne, HealthPlus, MediCareX, SecureLife"
4,chronic_flag,int8,True,,,"0, 1"
5,department,category,True,,,"Cardiology, ER, General, ICU, Neurology, Ortho..."
6,visit_type,category,True,,,"ER, ICU, OPD"
7,length_of_stay_hours,float32,True,0.0,720.0,
8,billed_amount,float32,True,0.0,1000000.0,
9,payment_days,float32,True,0.0,365.0,


## 2. Validation Demo — Patient Risk Model

### 2a. Valid payload (should pass)

In [3]:
valid_risk_payload = {
    "chronic_flag": 1,
    "gender": "F",
    "visit_frequency": 5,
}

ok, errors = validate_payload(valid_risk_payload, risk_schema)
print(f"Valid: {ok}")
print(f"Errors: {errors}")

Valid: True
Errors: []


### 2b. Check 1 — Missing required field

In [4]:
missing_field_payload = {
    "chronic_flag": 1,
    # gender is missing
    "visit_frequency": 3,
}

ok, errors = validate_payload(missing_field_payload, risk_schema)
print(f"Valid: {ok}")
for e in errors:
    print(f"  ✗ {e}")

Valid: False
  ✗ 'gender' is required but missing or null.


### 2c. Check 2 — Numeric range violation

In [5]:
range_violation_payload = {
    "chronic_flag": 1,
    "gender": "M",
    "visit_frequency": 0,   # must be >= 1
}

ok, errors = validate_payload(range_violation_payload, risk_schema)
print(f"Valid: {ok}")
for e in errors:
    print(f"  ✗ {e}")

Valid: False
  ✗ 'visit_frequency' value 0 is below minimum allowed value 1.


### 2d. Check 3 — Unseen category

In [6]:
unseen_category_payload = {
    "chronic_flag": 1,
    "gender": "X",           # not in ["M", "F"]
    "visit_frequency": 3,
}

ok, errors = validate_payload(unseen_category_payload, risk_schema)
print(f"Valid: {ok}")
for e in errors:
    print(f"  ✗ {e}")

Valid: False
  ✗ 'gender' value 'X' is not in the allowed set: ['M', 'F'].


## 3. Validation Demo — Claim Outcome Model

### 3a. Valid payload

In [7]:
valid_claim_payload = {
    "age": 45,
    "gender": "M",
    "city": "Mumbai",
    "insurance_provider": "HealthPlus",
    "chronic_flag": 0,
    "department": "General",
    "visit_type": "OPD",
    "length_of_stay_hours": 4.5,
    "billed_amount": 12000.0,
    "payment_days": 14.0,
    "visit_frequency": 3,
    "lag_time": 5,
}

ok, errors = validate_payload(valid_claim_payload, claim_schema)
print(f"Valid: {ok}")
print(f"Errors: {errors}")

Valid: True
Errors: []


### 3b. Multiple violations at once

In [8]:
multi_violation_payload = {
    "age": 200,                        # exceeds max_value=120
    "gender": "M",
    "city": "Singapore",               # unseen category
    "insurance_provider": "HealthPlus",
    "chronic_flag": 0,
    "department": "General",
    "visit_type": "OPD",
    "length_of_stay_hours": -1.0,      # below min_value=0
    # billed_amount is missing (required)
    "payment_days": 14.0,
    "visit_frequency": 3,
    "lag_time": 5,
}

ok, errors = validate_payload(multi_violation_payload, claim_schema)
print(f"Valid: {ok}")
for e in errors:
    print(f"  ✗ {e}")

Valid: False
  ✗ 'age' value 200 exceeds maximum allowed value 120.
  ✗ 'city' value 'Singapore' is not in the allowed set: ['Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Mumbai', 'Pune'].
  ✗ 'length_of_stay_hours' value -1.0 is below minimum allowed value 0.0.
  ✗ 'billed_amount' is required but missing or null.


## 4. Validation Rules Summary Table

In [9]:
all_rules = []
for model_name, schema in [("patient_risk", risk_schema), ("claim_outcome", claim_schema)]:
    for feat in schema["features"]:
        c = feat.get("constraints", {})
        checks = []
        if feat.get("required"):
            checks.append("not-null")
        if "min_value" in c or "max_value" in c:
            bounds = []
            if "min_value" in c:
                bounds.append(f">= {c['min_value']}")
            if "max_value" in c:
                bounds.append(f"<= {c['max_value']}")
            checks.append("range: " + ", ".join(bounds))
        if "allowed_values" in c:
            checks.append(f"in {c['allowed_values']}")
        all_rules.append({
            "Model": model_name,
            "Feature": feat["name"],
            "Type": feat["type"],
            "Validation Rules": " | ".join(checks) if checks else "type check only",
        })

rules_df = pd.DataFrame(all_rules)
display(rules_df)

,Model,Feature,Type,Validation Rules
0,patient_risk,chronic_flag,int8,"not-null | in [0, 1]"
1,patient_risk,gender,category,"not-null | in ['M', 'F']"
2,patient_risk,visit_frequency,int8,"not-null | range: >= 1, <= 100"
3,claim_outcome,age,int8,"not-null | range: >= 0, <= 120"
4,claim_outcome,gender,category,"not-null | in ['M', 'F']"
5,claim_outcome,city,category,"not-null | in ['Bangalore', 'Chennai', 'Delhi'..."
6,claim_outcome,insurance_provider,category,"not-null | in ['CareOne', 'HealthPlus', 'MediC..."
7,claim_outcome,chronic_flag,int8,"not-null | in [0, 1]"
8,claim_outcome,department,category,"not-null | in ['Cardiology', 'ER', 'General', ..."
9,claim_outcome,visit_type,category,"not-null | in ['ER', 'ICU', 'OPD']"


## 5. Prediction Audit Log Inspection

The API logs every prediction to `Phase 5/logs/predictions.log`. Each line contains:
- `prediction_id` — UUID for traceability
- `model` + version — model identity
- `ts` — UTC ISO-8601 timestamp
- `hash` — SHA-256 of input features (privacy-preserving)
- `label` — predicted class
- `score` — model confidence

In [10]:
if LOG_PATH.exists():
    log_lines = LOG_PATH.read_text().strip().splitlines()
    prediction_lines = [l for l in log_lines if "PREDICTION" in l]
    print(f"Total prediction log entries: {len(prediction_lines)}")
    print("\nSample entries (last 5):")
    for line in prediction_lines[-5:]:
        print(" ", line)
else:
    print(f"Log file not found at: {LOG_PATH}")
    print("Start the API server and make a prediction first.")

Total prediction log entries: 46

Sample entries (last 5):
  2026-05-01 20:37:31,488  INFO      PREDICTION | id=051cf458-afe7-4c77-8ccc-90a72b1dec5c | model=claim_outcome_model v1.0 | ts=2026-05-01T15:07:31.488289+00:00 | hash=4725f7104ef5fadcf94aebe5ab82c2caba85e93cb94a9fa0a565042debc4a806 | label=Pending | score=0.6030
  2026-05-01 20:37:38,286  INFO      PREDICTION | id=81f44870-7578-4ffd-9b34-27aea9969a71 | model=claim_outcome_model v1.0 | ts=2026-05-01T15:07:38.286455+00:00 | hash=50207bd27bb8974ea3b9c29940ccd83653c828973ee301c4e2eae30e1e03926b | label=Pending | score=0.7683
  2026-05-01 20:37:42,041  INFO      PREDICTION | id=cb641b72-7b19-40ff-bf2c-a6fa5c5ca5c8 | model=claim_outcome_model v1.0 | ts=2026-05-01T15:07:42.041129+00:00 | hash=5ac97ba7b4dfd1718b4d488b1bc63009ee58c8e4c2a2b5dc2b68424751e00019 | label=Paid | score=0.6291
  2026-05-01 21:01:02,898  INFO      PREDICTION | id=0f169498-4b49-4c88-a551-9dfa4d274e66 | model=patient_risk_model v1.0 | ts=2026-05-01T15:31:02.89848

In [11]:
if LOG_PATH.exists() and prediction_lines:
    pattern = re.compile(
        r"PREDICTION \| id=(?P<id>[\w-]+) \| model=(?P<model>[\w_]+) v(?P<version>[\d.]+) "
        r"\| ts=(?P<ts>[\S]+) \| hash=(?P<hash>\w+) \| label=(?P<label>\w+) \| score=(?P<score>[\d.]+)"
    )
    records = []
    for line in prediction_lines:
        m = pattern.search(line)
        if m:
            records.append(m.groupdict())

    log_df = pd.DataFrame(records)
    log_df["score"] = log_df["score"].astype(float)
    log_df["ts"] = pd.to_datetime(log_df["ts"])

    print("Parsed audit log:")
    display(log_df.head(10))

    print("\nPrediction distribution by model and label:")
    display(log_df.groupby(["model", "label"]).size().reset_index(name="count"))

    print("\nAverage confidence by model:")
    display(log_df.groupby("model")["score"].mean().reset_index())

Parsed audit log:


,id,model,version,ts,hash,label,score
0,56fc596d-3f25-4d86-9131-0ff84bb9f30f,patient_risk_model,1.0,2026-05-01 14:48:18.796775+00:00,aecbd702a18fcbdac8a02551be153304a03d18c1c50830...,Medium,0.3521
1,fda3d0d2-e028-46b9-bce4-926ab4071c61,claim_outcome_model,1.0,2026-05-01 14:48:23.914028+00:00,9b9f9a102aabc4baf8bc097a77606b8e57e9f8a6ece0b3...,Pending,0.6030
2,53ecc19e-60d8-4923-9eed-68dc149d9a50,claim_outcome_model,1.0,2026-05-01 14:49:10.906807+00:00,51d5f0643f05020ea70b6eda9b631a05e70e45b50f863e...,Pending,0.6030
3,d0490d4a-f377-484a-be2f-72420268037f,claim_outcome_model,1.0,2026-05-01 14:53:25.054208+00:00,cdda53b0b6662db8b1638fbc29449c9174e24dd9ded7c6...,Pending,0.6030
4,bcd1303d-8243-4eac-af95-37f88b743b17,claim_outcome_model,1.0,2026-05-01 14:53:28.200708+00:00,e07454ba7bc44863eee99ec5d5927c90eb1d194d7db743...,Pending,0.6117
5,84c459bb-270e-45a1-9f6e-0d5b430628b7,claim_outcome_model,1.0,2026-05-01 14:53:31.027373+00:00,266cbb9caa70319be08ec7493c957943c33e589b8ae8f4...,Pending,0.6117
6,10ea66ff-4baf-421e-8e45-e2b173aae97f,claim_outcome_model,1.0,2026-05-01 14:53:31.943781+00:00,266cbb9caa70319be08ec7493c957943c33e589b8ae8f4...,Pending,0.6117
7,619a82e7-f3e4-49a3-bd39-bf149f91b4e0,claim_outcome_model,1.0,2026-05-01 14:53:32.962131+00:00,266cbb9caa70319be08ec7493c957943c33e589b8ae8f4...,Pending,0.6117
8,3ab4bc23-822e-4d91-9e27-eae1627cb3cd,claim_outcome_model,1.0,2026-05-01 14:53:39.916686+00:00,e299c9af5857a70644be290c907e729776ab6b91a67e33...,Pending,0.6263
9,c10ddf34-9944-4ff7-9c93-e78c806085d5,claim_outcome_model,1.0,2026-05-01 14:53:42.922360+00:00,1d0944578de4716e4afaf018f9bf21efdfab5267512a42...,Pending,0.5847



Prediction distribution by model and label:


,model,label,count
0,claim_outcome_model,Paid,1
1,claim_outcome_model,Pending,29
2,claim_outcome_model,Rejected,1
3,patient_risk_model,High,4
4,patient_risk_model,Low,3
5,patient_risk_model,Medium,8



Average confidence by model:


,model,score
0,claim_outcome_model,0.614068
1,patient_risk_model,0.374187


## Summary

| Check Type | Scope | Behaviour |
|---|---|---|
| Missing / null | All required features | HTTP 422 with field name |
| Numeric range | `int8`, `int32`, `float32` features with bounds | HTTP 422 with value and bounds |
| Unseen category | All `category` and `int8` flag features with `allowed_values` | HTTP 422 with value and allowed set |

All violations are collected before responding, so a single bad request surfaces **all** errors at once. The audit log captures every successful prediction with a SHA-256 feature hash, enabling traceability without storing raw PII.